# Nocturne Aegis Cel Candidate Generator v4

This is the **a Colab-first iteration notebook** for `Nocturne Aegis Cel`.

Its purpose is to generate a **small, high-signal first pass of 3 images** for `Nocturne Aegis Cel`, to improve the base prompt to generate training candidates.

**Changelog:**

- `v1` preserved more of the style language, but the prose prompt was too long and got truncated by CLIP.
- `v2` fixed several SDXL issues, but leaned too far toward generic tag cleanup and still risked weak style steering.
- `v3` protected token length, but became too light on notebook guidance and silently trimmed prompts, which can hide failures.

This notebook balances those tradeoffs:

- clear Colab notes and documentation
- `StableDiffusionXLPipeline` with SDXL prompt channels
- compact style bundles that fit token limits
- explicit token reporting before generation
- **fail-fast** prompt validation instead of silent trimming by default
- a small default run: **3 prompts x 1 seed = 3 images**

After the 3-image pass looks acceptable, this notebook can be scaled into broader candidate generation for LoRA dataset curation.


## Working Principle

The base model does **not** know `Nocturne Aegis Cel` yet. Before the LoRA exists, the style must be expressed as **mechanical visual instructions** the base checkpoint can understand:

- 1990s cel-animation structure
- modern volumetric finish
- sharp geometric face design
- glass-like eyes
- jewel-toned palette
- layered cel shadows
- ambient occlusion
- iridescent metal and matte cloth separation

That style signal has to be strong enough to matter, but short enough to survive CLIP. The notebook therefore splits prompting across:

- `prompt`: quality tags + content tags + short style anchor
- `prompt_2`: compact global rendering/style rules
- `negative_prompt` and `negative_prompt_2`: explicit blockers for sketch, monochrome, unfinished output, anatomy errors, and clutter


In [ ]:
# Keep Colab-compatible versions for packages Colab already depends on.
# Upgrading pandas to 3.x or Pillow to 12.x causes resolver conflicts with the base runtime.
!pip install -q pandas==2.2.2 "Pillow<12.0"
!pip install -q -U diffusers transformers accelerate safetensors tqdm huggingface_hub

In [ ]:
import gc
import inspect
import json
import os
import zipfile
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

from diffusers import (
    DPMSolverMultistepScheduler,
    EulerAncestralDiscreteScheduler,
    StableDiffusionXLImg2ImgPipeline,
    StableDiffusionUpscalePipeline,
    StableDiffusionXLPipeline,
)


## Optional Colab Setup

Use these before model download.

- Mount Google Drive if you want outputs to persist across runtime resets.
- Prefer setting a Hugging Face token so Colab downloads authenticate cleanly and avoid anonymous rate limits.
- This notebook checks `HF_TOKEN`, then `HUGGINGFACEHUB_API_TOKEN`, then an optional Colab Secret named `HF_TOKEN`.


In [ ]:
# Optional: mount Google Drive for persistent storage.
# from google.colab import drive
# drive.mount('/content/drive')

# Hugging Face token setup.
# Recommended in Colab: open the Secrets panel and add HF_TOKEN there.
# Fallbacks: set HF_TOKEN or HUGGINGFACEHUB_API_TOKEN as environment variables.

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACEHUB_API_TOKEN")

if HF_TOKEN is None:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("Hugging Face token detected.")
else:
    print("No Hugging Face token detected. Public downloads may be rate limited.")
    print("Set HF_TOKEN in Colab Secrets or as an environment variable before loading the model.")

## Configuration

This notebook now defaults to an **Illustrious v0.1 tag-mode setup** instead of a hybrid natural-language setup.

Why:

- `Illustrious v0.1` is a base model trained primarily on tag-style conditioning.
- The Onoma report explicitly says `v0.1` struggles with longer natural-language prompts.
- The previous `Nocturne Aegis Cel` prose steering was not just weak; it produced broken outputs.

Recommended first pass:

- tag-based prompts
- `NUM_IMAGES_PER_PROMPT = 1`
- `NUM_INFERENCE_STEPS = 28`
- `GUIDANCE_SCALE = 6.5`
- `SCHEDULER = "dpmpp_2m_karras"` for a more aesthetic first-stage sample
- `CLIP_SKIP = None` unless you have evidence the chosen checkpoint benefits from it

If the model download is failing or throttled, stop here and set `HF_TOKEN` before running the load-model cell.


In [ ]:
# =========================
# CONFIG
# =========================

# Base model is configurable because v0.1 is useful for open LoRA training,
# but it is not the prettiest checkpoint by default.
MODEL_ID = "OnomaAIResearch/Illustrious-XL-v2.0"
PROMPTS_FILE = Path("src/prompts_illustrious_v4_tag_iteration.json")

# Change this to a Drive path if you mounted Drive.
OUTPUT_ROOT = Path("outputs/nocturne_aegis_candidates_v4")
IMAGES_DIR = OUTPUT_ROOT / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# Small first pass: 3 prompts x 1 image each.
NUM_IMAGES_PER_PROMPT = 1
BASE_SEED = 250505

# Onoma guidance: Euler A 20-28 / CFG 5-7.5 works,
# but their report also notes DPM-based schedulers can work better for aesthetic detailed setups.
NUM_INFERENCE_STEPS = 28
GUIDANCE_SCALE = 6.5
GUIDANCE_RESCALE = 0.0
CLIP_SKIP = None

# Options: "dpmpp_2m_karras", "euler_a", "default"
SCHEDULER = "dpmpp_2m_karras"

# v0.1 is tag-first. Keep prompt_2 optional and off by default.
USE_PROMPT_2 = False

# Controlled sweep for quality tuning. Keep small for Colab.
ENABLE_SWEEP = False
SWEEP_SCHEDULERS = [SCHEDULER]
SWEEP_STEPS = [NUM_INFERENCE_STEPS]
SWEEP_GUIDANCE_SCALES = [GUIDANCE_SCALE]
SWEEP_USE_PROMPT_2 = [USE_PROMPT_2]

# Optional SDXL refiner stage for higher local detail and finish.
USE_REFINER = False
REFINER_MODEL_ID = "stabilityai/stable-diffusion-xl-refiner-1.0"
# Options: "ensemble", "base_to_refiner"
REFINER_MODE = "ensemble"
REFINER_HIGH_NOISE_FRAC = 0.7
REFINER_STEPS = NUM_INFERENCE_STEPS
REFINER_GUIDANCE_SCALE = GUIDANCE_SCALE

# Optional final super-resolution stage.
# Hugging Face Diffusers provides an official x4 upscaler pipeline.
USE_FINAL_UPSCALER = False
FINAL_UPSCALER_MODEL_ID = "stabilityai/stable-diffusion-x4-upscaler"
FINAL_UPSCALER_STEPS = 75
FINAL_UPSCALER_GUIDANCE_SCALE = 9.0
FINAL_UPSCALER_NOISE_LEVEL = 20
SAVE_PRE_UPSCALE_IMAGE = True

# T4: True. L4 / A100: False is usually faster.
USE_CPU_OFFLOAD = True

# Prompt validation behavior.
MAX_TOKENS_PER_CHANNEL = 75
FAIL_ON_TOKEN_OVERFLOW = True

ASPECT_SIZES = {
    "portrait": (896, 1152),
    "full_body": (832, 1216),
    "wide": (1216, 832),
    "square": (1024, 1024),
}

print("Model:", MODEL_ID)
print("Prompt file:", PROMPTS_FILE)
print("Scheduler:", SCHEDULER)
print("USE_PROMPT_2:", USE_PROMPT_2)
print("ENABLE_SWEEP:", ENABLE_SWEEP)
print("USE_REFINER:", USE_REFINER)
print("USE_FINAL_UPSCALER:", USE_FINAL_UPSCALER)
print("Output root:", OUTPUT_ROOT)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Prompt Set

The default prompt file contains **3 deliberately different subjects**:

1. portrait / face-read test
2. full-body figure / silhouette test
3. mech-environment / hard-surface test

This is more useful than generating 3 near-duplicate characters. If these three images fail, scaling to 80 images will only scale the failure.


In [ ]:
if not PROMPTS_FILE.exists():
    raise FileNotFoundError(f"Prompt file not found: {PROMPTS_FILE}")

with open(PROMPTS_FILE, "r", encoding="utf-8") as f:
    prompt_rows = json.load(f)

assert len(prompt_rows) == 3, f"Expected 3 iteration prompts, found {len(prompt_rows)}"
for row in prompt_rows:
    assert row["caption"].startswith("nacel_v1,"), f"Caption must start with nacel_v1: {row['id']}"

pd.DataFrame(prompt_rows)

## Prompt Architecture

This notebook now uses a **tag-oriented prompt format** tuned for `Illustrious v0.1`.

That means:

- use model-native tags like `1girl`, `solo`, `upper body`, `mecha`, `ruined city`
- avoid long prose style descriptions during candidate generation
- do not use the untrained token `Nocturne Aegis Cel` before the LoRA exists
- keep composition tags simple and non-conflicting

This is intentionally less poetic and more practical. The goal at this stage is to get usable source images, not to describe the final style manifesto inside the prompt.


In [ ]:
# Illustrious v0.1 responds better to concise tag conditioning than to long prose.
# Keep prompts closer to Danbooru-style concepts and quality tokens.

QUALITY_TAGS = (
    "masterpiece, best quality, general"
)

STYLE_TAGS = (
    "anime coloring, cel shading, dramatic lighting, rim lighting, high contrast, detailed eyes"
)

# Optional second SDXL prompt channel. Keep tag-like and short when enabled.
PROMPT_2_TAGS = (
    "fantasy, science fiction, reflective metal, dark theme, volumetric lighting"
)

NEGATIVE_PROMPT = (
    "worst quality, bad quality, low quality, lowres, displeasing, very displeasing, comic, multiple views, "
    "bad anatomy, bad hands, scan artifacts, monochrome, greyscale, signature, twitter username, jpeg artifacts, "
    "2koma, 4koma, extra digits, fewer digits"
)

NEGATIVE_PROMPT_2 = (
    "worst quality, bad quality, low quality, blurry, monochrome, greyscale, bad anatomy, bad hands"
)


def build_positive_prompt(content_tags: str) -> str:
    return f"{QUALITY_TAGS}, {content_tags}, {STYLE_TAGS}"


## Load Model

Key changes from the previous broken setup:

- still uses `StableDiffusionXLPipeline`
- defaults to `DPM++ 2M Karras` instead of forcing `Euler A`
- leaves `clip_skip` off by default
- keeps the model open and configurable, but assumes `Illustrious v0.1` should be prompted in tags first

If you later switch to a more natural-language-capable checkpoint, you can enable `USE_PROMPT_2` and push prompt detail further.


In [ ]:
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

load_kwargs = dict(
    torch_dtype=torch_dtype,
    use_safetensors=True,
)
if HF_TOKEN:
    load_kwargs["token"] = HF_TOKEN

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    **load_kwargs,
)
refiner_pipe = None
upscaler_pipe = None

def apply_scheduler(target_pipe, scheduler_name):
    if scheduler_name == "euler_a":
        target_pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(target_pipe.scheduler.config)
    elif scheduler_name == "dpmpp_2m_karras":
        target_pipe.scheduler = DPMSolverMultistepScheduler.from_config(
            target_pipe.scheduler.config,
            use_karras_sigmas=True,
            algorithm_type="dpmsolver++",
        )
    elif scheduler_name != "default":
        raise ValueError(f"Unsupported scheduler: {scheduler_name}")

apply_scheduler(pipe, SCHEDULER)

if USE_REFINER:
    refiner_load_kwargs = dict(load_kwargs)
    refiner_pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
        REFINER_MODEL_ID,
        text_encoder_2=pipe.text_encoder_2,
        vae=pipe.vae,
        **refiner_load_kwargs,
    )
    apply_scheduler(refiner_pipe, SCHEDULER)

if USE_FINAL_UPSCALER:
    upscaler_load_kwargs = dict(load_kwargs)
    upscaler_pipe = StableDiffusionUpscalePipeline.from_pretrained(
        FINAL_UPSCALER_MODEL_ID,
        **upscaler_load_kwargs,
    )

pipe.enable_vae_slicing()
try:
    pipe.enable_vae_tiling()
except Exception:
    pass

if refiner_pipe is not None:
    refiner_pipe.enable_vae_slicing()
    try:
        refiner_pipe.enable_vae_tiling()
    except Exception:
        pass

if upscaler_pipe is not None:
    upscaler_pipe.enable_vae_slicing()
    try:
        upscaler_pipe.enable_vae_tiling()
    except Exception:
        pass

if torch.cuda.is_available():
    if USE_CPU_OFFLOAD:
        pipe.enable_model_cpu_offload()
        if refiner_pipe is not None:
            refiner_pipe.enable_model_cpu_offload()
        if upscaler_pipe is not None:
            upscaler_pipe.enable_model_cpu_offload()
        print("Model CPU offload enabled.")
    else:
        pipe.to("cuda")
        if refiner_pipe is not None:
            refiner_pipe.to("cuda")
        if upscaler_pipe is not None:
            upscaler_pipe.to("cuda")
        print("Pipeline moved to CUDA.")
else:
    print("CUDA unavailable. Running on CPU will be very slow.")


## Token Validation

This notebook does **not** silently trim prompts by default.

That is intentional. Silent trimming can hide the exact reason a run drifts off-style. Instead, this cell reports token counts clearly and fails if a prompt channel is too long.

If you later decide you want auto-trimming, add it deliberately after you know which tags matter most.


In [ ]:
def token_count(text, tokenizer):
    ids = tokenizer(text, truncation=False, add_special_tokens=True).input_ids
    return len(ids)


def prompt_report(prompt, prompt_2, neg, neg_2):
    tok1 = pipe.tokenizer
    tok2 = pipe.tokenizer_2 if getattr(pipe, "tokenizer_2", None) is not None else pipe.tokenizer

    report = {
        "prompt_tokens": token_count(prompt, tok1),
        "prompt_2_tokens": token_count(prompt_2, tok2) if prompt_2 else 0,
        "negative_prompt_tokens": token_count(neg, tok1),
        "negative_prompt_2_tokens": token_count(neg_2, tok2) if neg_2 else 0,
    }
    return report


def validate_prompt_channels(prompt, prompt_2, neg, neg_2, max_tokens=MAX_TOKENS_PER_CHANNEL):
    report = prompt_report(prompt, prompt_2, neg, neg_2)
    print(report)
    too_long = {k: v for k, v in report.items() if v > max_tokens}
    if too_long and FAIL_ON_TOKEN_OVERFLOW:
        raise ValueError(f"Prompt channel exceeds {max_tokens} tokens: {too_long}")
    return report


sample_positive = build_positive_prompt(prompt_rows[0]["content_tags"])
sample_prompt_2 = PROMPT_2_TAGS if USE_PROMPT_2 else None
sample_neg_2 = NEGATIVE_PROMPT_2 if USE_PROMPT_2 else None
sample_report = validate_prompt_channels(
    sample_positive,
    sample_prompt_2,
    NEGATIVE_PROMPT,
    sample_neg_2,
)

print("\nSample prompt:\n", sample_positive)
print("\nPrompt 2 enabled:", USE_PROMPT_2)
if USE_PROMPT_2:
    print("\nPrompt 2:\n", PROMPT_2_TAGS)


## Generation Helpers

`call_pipe()` sends both SDXL prompt channels and checks whether `clip_skip` and `guidance_rescale` are supported by the installed pipeline version.


In [ ]:
def build_generation_plan():
    schedulers = SWEEP_SCHEDULERS if ENABLE_SWEEP else [SCHEDULER]
    steps_list = SWEEP_STEPS if ENABLE_SWEEP else [NUM_INFERENCE_STEPS]
    guidance_scales = SWEEP_GUIDANCE_SCALES if ENABLE_SWEEP else [GUIDANCE_SCALE]
    prompt_2_flags = SWEEP_USE_PROMPT_2 if ENABLE_SWEEP else [USE_PROMPT_2]

    plans = []
    for scheduler_name in schedulers:
        for steps in steps_list:
            for guidance_scale in guidance_scales:
                for use_prompt_2 in prompt_2_flags:
                    plans.append({
                        "scheduler": scheduler_name,
                        "num_inference_steps": steps,
                        "guidance_scale": guidance_scale,
                        "use_prompt_2": use_prompt_2,
                    })
    return plans


def call_pipe(prompt, prompt_2, negative_prompt, negative_prompt_2, width, height, seed, scheduler_name, num_inference_steps, guidance_scale):
    generator = torch.Generator(device="cuda" if torch.cuda.is_available() else "cpu").manual_seed(seed)
    apply_scheduler(pipe, scheduler_name)
    if refiner_pipe is not None:
        apply_scheduler(refiner_pipe, scheduler_name)

    kwargs = dict(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=width,
        height=height,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=generator,
        original_size=(width, height),
        target_size=(width, height),
        crops_coords_top_left=(0, 0),
    )

    if prompt_2:
        kwargs["prompt_2"] = prompt_2
    if negative_prompt_2:
        kwargs["negative_prompt_2"] = negative_prompt_2

    sig = inspect.signature(pipe.__call__).parameters
    if "clip_skip" in sig and CLIP_SKIP is not None:
        kwargs["clip_skip"] = CLIP_SKIP
    if "guidance_rescale" in sig:
        kwargs["guidance_rescale"] = GUIDANCE_RESCALE

    with torch.inference_mode():
        if refiner_pipe is None:
            return pipe(**kwargs).images[0]

        refiner_sig = inspect.signature(refiner_pipe.__call__).parameters
        refiner_kwargs = dict(
            prompt=prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=REFINER_STEPS,
            guidance_scale=REFINER_GUIDANCE_SCALE,
            generator=generator,
        )
        if prompt_2 and "prompt_2" in refiner_sig:
            refiner_kwargs["prompt_2"] = prompt_2
        if negative_prompt_2 and "negative_prompt_2" in refiner_sig:
            refiner_kwargs["negative_prompt_2"] = negative_prompt_2
        if "guidance_rescale" in refiner_sig:
            refiner_kwargs["guidance_rescale"] = GUIDANCE_RESCALE

        if REFINER_MODE == "ensemble":
            latent = pipe(
                **kwargs,
                denoising_end=REFINER_HIGH_NOISE_FRAC,
                output_type="latent",
            ).images
            refiner_kwargs["image"] = latent
            refiner_kwargs["denoising_start"] = REFINER_HIGH_NOISE_FRAC
            return refiner_pipe(**refiner_kwargs).images[0]

        if REFINER_MODE == "base_to_refiner":
            latent = pipe(
                **kwargs,
                output_type="latent",
            ).images
            refiner_kwargs["image"] = latent
            return refiner_pipe(**refiner_kwargs).images[0]

        raise ValueError(f"Unsupported refiner mode: {REFINER_MODE}")


def maybe_upscale_image(image, prompt, negative_prompt, seed):
    if upscaler_pipe is None:
        return image

    generator = torch.Generator(device="cuda" if torch.cuda.is_available() else "cpu").manual_seed(seed)
    upscaler_sig = inspect.signature(upscaler_pipe.__call__).parameters
    kwargs = dict(
        prompt=prompt,
        image=image,
        num_inference_steps=FINAL_UPSCALER_STEPS,
        guidance_scale=FINAL_UPSCALER_GUIDANCE_SCALE,
        noise_level=FINAL_UPSCALER_NOISE_LEVEL,
        negative_prompt=negative_prompt,
        generator=generator,
    )
    if "clip_skip" in upscaler_sig and CLIP_SKIP is not None:
        kwargs["clip_skip"] = CLIP_SKIP

    with torch.inference_mode():
        return upscaler_pipe(**kwargs).images[0]


def make_contact_sheet(image_paths, out_path, thumb_w=240, thumb_h=240, cols=3):
    image_paths = [Path(p) for p in image_paths]
    if not image_paths:
        return None

    rows = (len(image_paths) + cols - 1) // cols
    label_h = 34
    sheet = Image.new("RGB", (cols * thumb_w, rows * (thumb_h + label_h)), "white")
    draw = ImageDraw.Draw(sheet)

    for i, path in enumerate(image_paths):
        img = Image.open(path).convert("RGB")
        img.thumbnail((thumb_w, thumb_h))
        x = (i % cols) * thumb_w + (thumb_w - img.width) // 2
        y = (i // cols) * (thumb_h + label_h)
        sheet.paste(img, (x, y))
        draw.text((i % cols * thumb_w + 6, y + thumb_h + 6), path.stem[:32], fill=(0, 0, 0))

    sheet.save(out_path, quality=92)
    return out_path


## Preflight Check

This cell is now checking a **tag-mode Illustrious v0.1** setup.

Review:

- token counts
- exact tag string sent to the model
- whether `prompt_2` is disabled or enabled
- chosen resolution per prompt

If these look wrong, do not generate images yet.


In [ ]:
generation_plan = build_generation_plan()
print(f"Generation plan count: {len(generation_plan)}")

preflight_rows = []
for row in prompt_rows:
    aspect = row.get("aspect", "square")
    width, height = ASPECT_SIZES.get(aspect, ASPECT_SIZES["square"])
    positive = build_positive_prompt(row["content_tags"])

    for plan_idx, plan in enumerate(generation_plan, start=1):
        prompt_2 = PROMPT_2_TAGS if plan["use_prompt_2"] else None
        neg_2 = NEGATIVE_PROMPT_2 if plan["use_prompt_2"] else None
        report = validate_prompt_channels(positive, prompt_2, NEGATIVE_PROMPT, neg_2)
        preflight_rows.append({
            "id": row["id"],
            "plan_index": plan_idx,
            "aspect": aspect,
            "width": width,
            "height": height,
            **plan,
            "use_refiner": USE_REFINER,
            "refiner_mode": REFINER_MODE if USE_REFINER else "",
            **report,
            "positive_prompt": positive,
            "prompt_2": prompt_2 or "",
        })

preflight_df = pd.DataFrame(preflight_rows)
preflight_df


## Generate 3 Iteration Images

This run is now intended to answer a narrower question:

`Can Illustrious v0.1 produce coherent tag-driven source images for Nocturne Aegis candidate generation?`

If the result is still broken after this change, the next conclusion is not “tune the prompt more.” It is “switch checkpoints or add a second-stage refine workflow.”


In [ ]:
metadata = []
generated_paths = []

generation_plan = build_generation_plan()

for p_idx, row in enumerate(tqdm(prompt_rows, desc="Prompts"), start=1):
    aspect = row.get("aspect", "square")
    width, height = ASPECT_SIZES.get(aspect, ASPECT_SIZES["square"])
    positive = build_positive_prompt(row["content_tags"])

    for plan_idx, plan in enumerate(generation_plan, start=1):
        prompt_2 = PROMPT_2_TAGS if plan["use_prompt_2"] else None
        neg_2 = NEGATIVE_PROMPT_2 if plan["use_prompt_2"] else None
        report = validate_prompt_channels(positive, prompt_2, NEGATIVE_PROMPT, neg_2)

        for v_idx in range(NUM_IMAGES_PER_PROMPT):
            seed = BASE_SEED + (p_idx * 100000) + (plan_idx * 1000) + v_idx
            image = call_pipe(
                prompt=positive,
                prompt_2=prompt_2,
                negative_prompt=NEGATIVE_PROMPT,
                negative_prompt_2=neg_2,
                width=width,
                height=height,
                seed=seed,
                scheduler_name=plan["scheduler"],
                num_inference_steps=plan["num_inference_steps"],
                guidance_scale=plan["guidance_scale"],
            )

            stem = f"{row['id']}_p{plan_idx}_seed{seed}"
            image_path = IMAGES_DIR / f"{stem}.png"
            caption_path = IMAGES_DIR / f"{stem}.txt"

            base_image = image
            if SAVE_PRE_UPSCALE_IMAGE and USE_FINAL_UPSCALER:
                base_image_path = IMAGES_DIR / f"{stem}_base.png"
                base_image.save(base_image_path)
            else:
                base_image_path = None

            final_image = maybe_upscale_image(
                image=image,
                prompt=positive,
                negative_prompt=NEGATIVE_PROMPT,
                seed=seed,
            )

            final_image.save(image_path)
            caption_path.write_text(row["caption"].strip() + "\n", encoding="utf-8")
            generated_paths.append(image_path)

            metadata.append({
                "id": row["id"],
                "plan_index": plan_idx,
                "seed": seed,
                "aspect": aspect,
                "width": width,
                "height": height,
                "image": str(image_path),
                "caption_file": str(caption_path),
                "caption": row["caption"],
                "content_tags": row["content_tags"],
                "prompt": positive,
                "prompt_2": prompt_2 or "",
                "negative_prompt": NEGATIVE_PROMPT,
                "negative_prompt_2": neg_2 or "",
                **report,
                "num_inference_steps": plan["num_inference_steps"],
                "guidance_scale": plan["guidance_scale"],
                "clip_skip": CLIP_SKIP,
                "scheduler": plan["scheduler"],
                "use_prompt_2": plan["use_prompt_2"],
                "use_refiner": USE_REFINER,
                "refiner_model_id": REFINER_MODEL_ID if USE_REFINER else "",
                "refiner_mode": REFINER_MODE if USE_REFINER else "",
                "refiner_steps": REFINER_STEPS if USE_REFINER else "",
                "refiner_guidance_scale": REFINER_GUIDANCE_SCALE if USE_REFINER else "",
                "use_final_upscaler": USE_FINAL_UPSCALER,
                "final_upscaler_model_id": FINAL_UPSCALER_MODEL_ID if USE_FINAL_UPSCALER else "",
                "final_upscaler_steps": FINAL_UPSCALER_STEPS if USE_FINAL_UPSCALER else "",
                "final_upscaler_guidance_scale": FINAL_UPSCALER_GUIDANCE_SCALE if USE_FINAL_UPSCALER else "",
                "final_upscaler_noise_level": FINAL_UPSCALER_NOISE_LEVEL if USE_FINAL_UPSCALER else "",
                "pre_upscale_image": str(base_image_path) if base_image_path is not None else "",
                "final_width": final_image.width,
                "final_height": final_image.height,
                "generated_at_utc": datetime.utcnow().isoformat(),
            })

            display(final_image.resize((min(320, final_image.width), int(final_image.height * min(320, final_image.width) / final_image.width))))

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

metadata_df = pd.DataFrame(metadata)
metadata_df.to_csv(OUTPUT_ROOT / "metadata.csv", index=False)
metadata_df.to_json(OUTPUT_ROOT / "metadata.jsonl", orient="records", lines=True, force_ascii=False)
metadata_df


## Contact Sheet and Archive

The contact sheet is the fastest way to judge whether the run is worth iterating on.

Reject and revise if you see:

- sketch-like finish
- muddy or weak color
- poor face readability
- weak silhouette
- broken hands or anatomy
- generic anime drift
- missing material separation between skin, cloth, and metal


In [ ]:
contact_sheet_path = OUTPUT_ROOT / "contact_sheet_v4.jpg"
make_contact_sheet(generated_paths, contact_sheet_path, cols=3)

archive_path = OUTPUT_ROOT / "output_archive_v4.zip"
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if path.is_file() and path != archive_path:
            zf.write(path, path.relative_to(OUTPUT_ROOT))

print("Contact sheet:", contact_sheet_path)
print("Archive:", archive_path)
Image.open(contact_sheet_path)

## Next Iteration Rules

Interpret the next run like this:

1. If images become coherent but still generic, keep the checkpoint and tune tags.
2. If images become coherent but under-detailed, try `Euler A` or raise steps from `28` to `32`.
3. If images remain blob-like, stop blaming steps and switch checkpoint strategy.
4. If you later use a more capable Illustrious release, revisit `USE_PROMPT_2=True` and richer caption-like prompting.

For `Illustrious v0.1`, the highest-value fixes are:

- tag-native prompts
- known quality/rating tags like `general`
- simple composition tags
- 1MP-ish resolutions
- sampler experimentation between `DPM++ 2M Karras` and `Euler A`
